# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [1]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio
import datetime

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:50288/status,
Dashboard: http://127.0.0.1:50288/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:50291,Workers: 0
Dashboard: http://127.0.0.1:50288/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:49346,Total threads: 2
Dashboard: http://127.0.0.1:49347/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:50296,


# Value to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["Ihutai", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "LeftBank_Nov25", "ThePoint_Nov25"]
large_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua",
                "LeftBank_Nov25", "LeftBank_Feb26", "ThePoint_Nov25", "ThePoint_Feb26"]


In [4]:
satellite_classes_all = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3, 'Gracilaria submerged': 4, 
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_all_unet = {'Seagrass': 0, 'Seagrass submerged': 1, 'Gracilaria': 2, 'Gracilaria submerged': 3, 
                           'Ulva': 4, 'Submerged vegetation': 5, 'Microphytobenthos': 6, 'Satmarsh': 7,
                           'Unvegetated': 8, 'Water': 9, 'Terrestrial': 10, 'Rock': 11, 'Mixed': 12,}

In [27]:
sample_method = "sampling_2"
method_2_threshold = .8
model_max_cloud_cover = 5 # percentage
model_low_tide_delta_hrs = 0
model_low_tide_delta_mins = 30
test_threshold = 0.1

predict_max_cloud_cover = 1 # percentage"
predict_low_tide_delta_hrs = 2
predict_low_tide_delta_mins = 0
use_threshold=True

mndwi_file = data_path / "mndwi_low_tide_thresholds.csv"

model_description = f"RF_model_filtered_dates_13_class_{int(test_threshold*100)}_percent_test"
# f"RF_model_all_trained_13_class_{int(test_threshold*100)}_percent_test" f"RF_model_ihutai_trained_13_class_{int(test_threshold*100)}_percent_test" 

website_tab = (
    f"{model_description}_train_{utils.get_samples_folder(sample_method, method_2_threshold)}_"
    f"{utils.get_low_tide_max_cloud_name(low_tide_delta_hrs=model_low_tide_delta_hrs, low_tide_delta_mins=model_low_tide_delta_mins, max_cloud_cover=model_max_cloud_cover)}_"
    f"predict_{utils.get_low_tide_max_cloud_name(use_threshold=use_threshold, max_cloud_cover=predict_max_cloud_cover, low_tide_delta_hrs=predict_low_tide_delta_hrs, low_tide_delta_mins=predict_low_tide_delta_mins)}"
)

In [12]:
website_tab

'RF_model_filtered_dates_13_class_10_percent_test_train_sampling_2_80_percent_tide_30mins_cloud_5_predict_tide_thresholded_and_2hrs_cloud_1'

In [35]:
model_path = utils.get_models_path(sample_method=sample_method,
                                   method_2_threshold=method_2_threshold,
                                   low_tide_delta_hrs=model_low_tide_delta_hrs,
                                   low_tide_delta_mins=model_low_tide_delta_mins,
                                   max_cloud_cover=model_max_cloud_cover)
model_file = model_path / model_description / "model.joblib"
prediction_sites = all_training_sites 
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [39]:
model_path

WindowsPath('C:/Local/repos/seagrass-detection/data/models/tide_30mins_cloud_5/sampling_2_80_percent')

In [40]:
model_file.exists()

True

In [24]:
data_path = utils.get_data_path()
utils.create_data_folders()
predictions_path, satellite_path = utils.get_prediction_path(
    sample_method,
    method_2_threshold,
    model_max_cloud_cover=model_max_cloud_cover,
    predict_max_cloud_cover=predict_max_cloud_cover,
)

predictions_path = predictions_path / f"post_filter_{utils.get_low_tide_max_cloud_name(use_threshold=use_threshold, max_cloud_cover=predict_max_cloud_cover, low_tide_delta_hrs=predict_low_tide_delta_hrs, low_tide_delta_mins=predict_low_tide_delta_mins)}" / model_description
predictions_path.mkdir(exist_ok=True, parents=True)

In [25]:
predictions_path

WindowsPath('C:/Local/repos/seagrass-detection/data/predictions/cloud_1/model_sampling_2_80_percent_cloud_5/post_filter_tide_thresholded_and_2hrs_cloud_1/RF_model_filtered_dates_13_class_10_percent_test')

## Predict values

In [45]:
polygon_file

WindowsPath('C:/Local/repos/seagrass-detection/data/site_polygons/Ihutai_polygon.gpkg')

In [51]:
polygons = []
for site in all_training_sites:
    polygon_file = utils.get_site_polygon_path(site)
    polygons.append(geopandas.read_file(polygon_file))

In [52]:
import pandas
polygons_df = pandas.concat(polygons)

In [54]:
polygons_df.to_file(polygon_file.with_name("all_training_sites.gpkg"))
polygons_df.to_file(polygon_file.with_name("all_training_sites.shp"))                   

In [43]:
all_baselines = sentinel2.load_band_quality_baseline()
mndwi_thresholds = pandas.read_csv(mndwi_file)

## Format for website
website_folder = data_path / "website" / "mndwi_threshold" / website_tab
website_folder.mkdir(exist_ok=True, parents=True)
website_options = {"value": prediction_sites, "label": prediction_sites}
website_options = pandas.DataFrame(website_options)
website_options.to_csv(website_folder / "options.csv", index=False)

for prediction_site in prediction_sites:

    website_site_folder = website_folder / prediction_site
    website_site_folder.mkdir(exist_ok=True, parents=True)
    if (website_site_folder / "info_all_dates.csv").exists():
        print(f"{prediction_site} already predicted and prepared for website. Skipping. Delete if unexpected.")
        continue
    site_baseline = all_baselines.loc[all_baselines["site"] == prediction_site]
    mndwi_threshold = mndwi_thresholds.loc[mndwi_thresholds["site"] == prediction_site, "threshold",].squeeze()

    ## Predict values
    for year in years:
        print(f"{prediction_site}: year {year}")
        prediction_file = predictions_path / f"{prediction_site}_{year}.nc"
        polygon_file = utils.get_site_polygon_path(prediction_site)
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        
        if not prediction_file.exists():
            model_feature_names_file = model_file.with_name("feature_names.csv")
            if not satellite_file.exists():
                print("\tWARNING no satellite file to predict. Rerun the above cell if this is unexpected."
                      " Otherwise this indicates no suitable low-tide and low-cloud dates in the year.")
                continue
            predictions, satellite_data = training.predict_site_low_tide(
                test_satellite_file=satellite_file,
                polygon_file=polygon_file,
                model_file=model_file,
                low_tide_delta_hrs=predict_low_tide_delta_hrs,
                low_tide_delta_mins=predict_low_tide_delta_mins,
                mndwi_threshold=mndwi_threshold,
                anomaly_filter_baseline=site_baseline,
                model_feature_names_file=model_feature_names_file)
            if predictions is None:
                continue
            utils.save_netcdf(predictions, prediction_file)
        else:
            predictions = utils.load_classification(filename=prediction_file, chunks=None)
    
        for target_name in ["Seagrass", "Ulva", "Gracilaria"]:
            target_file = predictions_path /  f"{prediction_site}_{year}_{target_name.lower()}.gpkg"
            if not target_file.exists():
                target = {"date": [], "geometry": []}
                prediction = utils.load_classification(filename=prediction_file)
                prediction = utils.ensure_grid_indexing(prediction)
                target_id = pandas.read_csv(model_file.with_name("class_mappings.csv"), index_col=0)
                target_id = int(target_id.loc[target_id.index == target_name, "satellite_class_id"].squeeze())
                for time_index in range(len(prediction["time"])):
                    mask = predictions.isel(time=time_index) == target_id
                    target["date"].append(str(mask.time.data))
                    geometry = utils.mask_to_polygons(mask).dissolve().geometry
                    target["geometry"].append(geometry[0])
                target = geopandas.GeoDataFrame(target, crs="2193")
                target.to_file(target_file)

    if (website_site_folder / "seagrass_presence_absence_map.gpkg").exists():
        print(f"{prediction_site} already run. Go to next")
        continue
    extents_all_years = {"Seagrass": [], "Ulva": [], "Gracilaria": []}
    info = {"ids": [], "B04_scale":[], "B03_scale":[], "B02_scale": [], "date":[]}
    for year in years:
        print(f"{prediction_site}: year {year}")
        satellite_info_set = False
        for target in extents_all_years.keys():
            target_file = predictions_path / f"{prediction_site}_{year}_{target.lower()}.gpkg"

            if not target_file.exists():
                print(f"WARNING no {target_file.name} file. Run 'download_satellite_predictions.ipynb' is this is unexpected.")
                continue
            target_extents = geopandas.read_file(target_file)
            extents_all_years[target].append(target_extents)

            for index, row in target_extents.iterrows():
                target_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
                date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
        
                filename = f"{date_YYMMDD}_{target.lower()}.gpkg"
                target_extent.to_file(website_site_folder / filename)

                # Look up the satellite tile information
                if not satellite_info_set:
                    #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
                    site_polygon = geopandas.read_file(utils.get_site_polygon_path(prediction_site))
                    tile_id, precentage_dict = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
                    
                    info["date"].append(row.date)
                    info["ids"].append(tile_id)
                    info["B04_scale"].append(precentage_dict["B04"])
                    info["B03_scale"].append(precentage_dict["B03"])
                    info["B02_scale"].append(precentage_dict["B02"])
            satellite_info_set = True
            
    info = pandas.DataFrame(info)
    info = info.rename(columns={"ids": "Satellite Tile IDs",
                                "B04_scale": "B04 Rescale",
                                "B03_scale": "B03 Rescale",
                                "B02_scale": "B02 Rescale"})
    
    for key, value in extents_all_years.items():
        print(f"\tTarget {key}")
        target_extents = pandas.concat(value, ignore_index=True)
        target_extents['area'] = target_extents.area
        info[f"{key} Area [m^2]"] = target_extents.area
        target_extents.dissolve()[["geometry"]].to_file(website_site_folder / f"{key.lower()}_presence_absence_map.gpkg")

    
    info.to_csv(website_site_folder / "info_all_dates.csv", index=False)
                

Ihutai: year 2016
Ihutai: year 2017
Ihutai: year 2018
Ihutai: year 2019
Ihutai: year 2020
Ihutai: year 2021
Ihutai: year 2022
Ihutai: year 2023
	MNDWI filter: 2 of 9 dates kept
	Tide API filter: 0 of 2 dates kept
	WARNING - no dates survive the low-tide filters. Skipping predictions.
Ihutai: year 2024
	MNDWI filter: 4 of 21 dates kept
	Tide API filter: 3 of 4 dates kept
		2024-02-22: B04=0.55, B03=0.26, B02=0.29 -> kept
		2024-09-29: B04=0.06, B03=0.09, B02=0.15 -> kept
		2024-10-29: B04=0.24, B03=0.26, B02=0.17 -> kept
	Anomaly filter: 3 of 3 dates kept for site Ihutai (n_mad=3)
	Predict for 3 satellite images
	Predict satellite image for date 2024-02-22
	Predicted for date 2024-02-22
	Predict satellite image for date 2024-09-29
	Predicted for date 2024-09-29
	Predict satellite image for date 2024-10-29
	Predicted for date 2024-10-29
	Combine predictions and write conventions
Ihutai: year 2025
	MNDWI filter: 3 of 13 dates kept
	Tide API filter: 2 of 3 dates kept
		2025-01-27: B04=0.02